
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


#Lecture - Introduction to Multiplex Streaming, Delta Sinks, and Iceberg Reads

#### Duration: ~8-10 Minutes

This lecture introduces three advanced concepts used in Spark Declarative Pipelines: the **Multiplex pattern** for efficiently ingesting mixed event streams, **Delta Sinks** for writing streaming data to external tables, and **Iceberg reads via Delta UniForm** for enabling cross-platform access to Delta tables.
## Learning Objectives

By the end of this lecture, you will be able to:

- Explain the **Multiplex pattern** and how fan-out works for mixed event streams
- Describe what a **Delta Sink** is and when to use it over managed streaming tables
- Explain how **Delta UniForm** enables Iceberg reads without data duplication
- Understand how these three concepts **chain together** in a single pipeline

## A. The Multiplex Pattern

The Multiplex pattern addresses a common production challenge: efficiently processing multiple event types that arrive through a single data transport mechanism.

### A1. The Problem: One Stream, Many Schemas

In production environments, multiple business systems often share a **single data transport** such as:
- One Kafka topic
- One cloud storage path  
- One message queue

Each message carries a **type field** identifying which business domain it belongs to. Without the Multiplex pattern, this creates significant operational overhead:

<div style="flex-wrap:wrap;display:flex;gap:16px;margin:16px 0;">
<div style="flex:1 1 280px;min-width:0;background:#FFEBEE;border:1px solid #FFCDD2;border-radius:6px;padding:14px 16px;">
<div style="font-weight:bold;color:#C62828;font-size:13px;margin-bottom:8px;"> Without Multiplex</div>
<ul style="font-size:13px;color:#333;margin:0;padding-left:16px;">
<li>N event types → N separate pipelines</li>
<li>N separate checkpoints and source scans</li>
<li>Source change must be applied N times</li>
</ul>
</div>
<div style="flex:1 1 280px;min-width:0;background:#E8F5E9;border:1px solid #C8E6C9;border-radius:6px;padding:14px 16px;">
<div style="font-weight:bold;color:#1B5E20;font-size:13px;margin-bottom:8px;"> With Multiplex</div>
<ul style="font-size:13px;color:#333;margin:0;padding-left:16px;">
<li>N event types → 1 ingestion pipeline</li>
<li>1 checkpoint, 1 source scan shared across all domains</li>
<li>Source change applied in one place</li>
</ul>
</div>
</div>

**Reference Documentation:**
- [Multiplexing Data Pipelines Blog](https://www.databricks.com/blog/2022/04/27/how-uplift-built-cdc-and-multiplexing-data-pipelines-with-databricks-delta-live-tables.html)


### A2. Ingest Once, Fan Out by Type

The Multiplex pattern follows a simple but powerful approach:

1. **Single Ingestion**: Read all event types into one bronze table
2. **Type-Based Filtering**: Use the event type field to separate domains
3. **Fan-Out Processing**: Create domain-specific tables downstream

**Architecture Flow:**
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 330" width="100%" role="img" aria-label="A mixed event stream is ingested once into a bronze table, then fanned out by type into domain tables" style="font-family:'DM Sans',Arial,Helvetica,sans-serif;max-width:960px;display:block;margin:12px auto;"><defs><filter id="fo-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter><marker id="fo-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#5A6F77"/></marker><marker id="fo-ah-l" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF3621"/></marker><marker id="fo-ah-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#2272B4"/></marker><marker id="fo-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="fo-ah-k" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker></defs><rect x="10" y="100" width="180" height="120" rx="14" fill="#5A6F77" stroke="#3F535B" stroke-width="2" filter="url(#fo-sh)"/><g transform="translate(68,112) scale(1.0)"><path d="M14,40 a14,14 0 0 1 3,-27 a18,18 0 0 1 34,-4 a15,15 0 0 1 12,31 z" fill="#fff" stroke="#fff" stroke-width="2.4" stroke-linejoin="round"/></g><text x="100" y="184" font-size="16" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Mixed Event Stream</text><text x="100" y="205" font-size="13" font-weight="400" fill="#E3EAEE" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >one shared transport</text><path d="M192,160 H316" fill="none" stroke="#1B3139" stroke-width="2.6" marker-end="url(#fo-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><text x="254" y="144" font-size="13.5" font-weight="700" fill="#1B3139" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Single ingest</text><text x="254" y="186" font-size="13" font-weight="400" fill="#5A6F77" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >one checkpoint</text><rect x="320" y="80" width="250" height="160" rx="12" fill="#FF3621" stroke="#C92A18" stroke-width="2" filter="url(#fo-sh)"/><rect x="332" y="92" width="36" height="36" rx="8.64" fill="#fff" stroke="none" stroke-width="0" filter="url(#fo-sh)"/><svg x="337.76" y="98.98400000000001" width="24.48" height="22.3" viewBox="0 0 56 51" fill="#FF5F46"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/></svg><text x="445" y="138" font-size="20" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Bronze Table</text><text x="445" y="164" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >All event types together</text><rect x="362" y="186" width="166" height="34" rx="17" fill="#C92A18" stroke="none" stroke-width="0" /><text x="445" y="208" font-size="14" font-weight="600" fill="#fff" text-anchor="middle" style="font-family:Consolas,Monaco,monospace" >VARIANT payload</text><rect x="700" y="22" width="250" height="76" rx="12" fill="#EF6C00" stroke="#B84F00" stroke-width="2" filter="url(#fo-sh)"/><rect x="712" y="40.0" width="40" height="40" rx="9.6" fill="#fff" stroke="none" stroke-width="0" filter="url(#fo-sh)"/><svg x="718.4" y="47.76" width="27.2" height="24.8" viewBox="0 0 56 51" fill="#FF5F46"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/></svg><text x="851.0" y="54.0" font-size="18" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Domain A Table</text><text x="851.0" y="76.0" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:Consolas,Monaco,monospace" >WHERE type = A</text><path d="M572,160 C640,160 630,60 698,60" fill="none" stroke="#1B3139" stroke-width="2.4" marker-end="url(#fo-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="700" y="132" width="250" height="76" rx="12" fill="#EF6C00" stroke="#B84F00" stroke-width="2" filter="url(#fo-sh)"/><rect x="712" y="150.0" width="40" height="40" rx="9.6" fill="#fff" stroke="none" stroke-width="0" filter="url(#fo-sh)"/><svg x="718.4" y="157.76000000000002" width="27.2" height="24.8" viewBox="0 0 56 51" fill="#FF5F46"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/></svg><text x="851.0" y="164.0" font-size="18" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Domain B Table</text><text x="851.0" y="186.0" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:Consolas,Monaco,monospace" >WHERE type = B</text><path d="M572,160 C640,160 630,170 698,170" fill="none" stroke="#1B3139" stroke-width="2.4" marker-end="url(#fo-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="700" y="242" width="250" height="76" rx="12" fill="#EF6C00" stroke="#B84F00" stroke-width="2" filter="url(#fo-sh)"/><rect x="712" y="260.0" width="40" height="40" rx="9.6" fill="#fff" stroke="none" stroke-width="0" filter="url(#fo-sh)"/><svg x="718.4" y="267.76" width="27.2" height="24.8" viewBox="0 0 56 51" fill="#FF5F46"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/></svg><text x="851.0" y="274.0" font-size="18" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Domain C Table</text><text x="851.0" y="296.0" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:Consolas,Monaco,monospace" >WHERE type = C</text><path d="M572,160 C640,160 630,280 698,280" fill="none" stroke="#1B3139" stroke-width="2.4" marker-end="url(#fo-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><text x="445" y="272" font-size="14" font-weight="400" fill="#5A6F77" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Fan-out: one filtered table per domain</text></svg>

**Key Benefits:**
- Single checkpoint management
- Shared source scanning
- Centralized error handling
- Simplified monitoring


## B. Sinks

Sinks provide a mechanism to write streaming data from a Spark Declarative Pipeline to external Delta tables that exist outside the pipeline's managed scope.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 330" width="100%" role="img" aria-label="A pipeline flow writes either to a managed streaming table or materialized view, or through append_flow to an external sink" style="font-family:'DM Sans',Arial,Helvetica,sans-serif;max-width:960px;display:block;margin:12px auto;"><defs><filter id="sk-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter><marker id="sk-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#5A6F77"/></marker><marker id="sk-ah-l" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF3621"/></marker><marker id="sk-ah-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#2272B4"/></marker><marker id="sk-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="sk-ah-k" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker></defs><rect x="10" y="125" width="200" height="80" rx="12" fill="#FF3621" stroke="#C92A18" stroke-width="2" filter="url(#sk-sh)"/><rect x="22" y="145.0" width="40" height="40" rx="9.6" fill="#fff" stroke="none" stroke-width="0" filter="url(#sk-sh)"/><svg x="28.4" y="151.4" width="27.2" height="27.2" viewBox="0 0 150 150" fill="none"><path d="M134.08 35.53C129.52 35.53 125.73 38.86 125.01 43.21H111.85V27.53H108.85V59.69H103.02L103.01 52.91H101.51C91.51 52.91 83.38 61.04 83.38 71.04V72.54L88 72.55V77.46L84.2 77.47V78.97C84.2 88.51 80.48 97.48 73.74 104.24C67.34 110.63 58.95 114.3 49.97 114.67V106.74C59.85 106.08 76.27 98.89 76.27 75.01C76.27 46.63 100.49 46.24 101.52 46.23H103.02V32.31H101.52C91.18 32.31 81.45 36.34 74.13 43.65C67.17 50.62 63.19 59.76 62.82 69.54L58.99 69.55V80.44L63.54 80.45C62.83 87.6 57.13 93.3 49.97 94.01V87.3H38.14V103.77H24.99C24.27 99.42 20.48 96.09 15.92 96.09C10.86 96.09 6.73999 100.21 6.73999 105.27C6.73999 110.34 10.86 114.46 15.92 114.46C17.38 114.46 18.82 114.11 20.09 113.45L18.71 110.79C17.86 111.23 16.9 111.46 15.92 111.46C12.51 111.46 9.73999 108.68 9.73999 105.27C9.73999 101.86 12.51 99.09 15.92 99.09C19.33 99.09 22.11 101.86 22.11 105.27V106.77H38.14V122.46H41.14V90.31H46.97L46.98 97.09H48.48C58.48 97.09 66.61 88.96 66.61 78.96V77.46L61.99 77.45V72.54L65.79 72.53V71.03C65.79 61.49 69.5 52.52 76.25 45.76C82.65 39.37 91.04 35.7 100.02 35.33V43.31C94.18 43.89 73.27 47.84 73.27 75C73.27 103.38 49.48 103.78 48.47 103.78H46.97V117.69H48.47C58.81 117.69 68.54 113.66 75.86 106.35C82.82 99.38 86.8 90.24 87.17 80.46L91 80.45V69.55L86.45 69.54C87.16 62.39 92.86 56.69 100.02 55.98V62.69H111.85V46.21H127.89V44.71C127.89 41.3 130.67 38.53 134.08 38.53C137.49 38.53 140.26 41.3 140.26 44.71C140.26 48.12 137.49 50.9 134.08 50.9C133.1 50.9 132.14 50.67 131.29 50.23L129.91 52.89C131.18 53.55 132.62 53.9 134.08 53.9C139.14 53.9 143.26 49.78 143.26 44.71C143.26 39.65 139.14 35.53 134.08 35.53Z" fill="#FF5F46"/></svg><text x="136.0" y="159.0" font-size="18" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Pipeline Flow</text><text x="136.0" y="181.0" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:Consolas,Monaco,monospace" >append_flow</text><rect x="440" y="14" width="510" height="134" rx="14" fill="#FFF4D6" stroke="#FFAB00" stroke-width="2" /><text x="458" y="40" font-size="14.5" font-weight="700" fill="#B36B00" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Inside Pipeline — Managed Scope</text><rect x="440" y="182" width="510" height="134" rx="14" fill="#E6F0F8" stroke="#2272B4" stroke-width="2" /><text x="458" y="208" font-size="14.5" font-weight="700" fill="#0D47A1" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Outside Pipeline — External Targets</text><rect x="470" y="56" width="450" height="70" rx="12" fill="#fff" stroke="#EF6C00" stroke-width="2" filter="url(#sk-sh)"/><rect x="482" y="71.0" width="40" height="40" rx="9.6" fill="#fff" stroke="none" stroke-width="0" filter="url(#sk-sh)"/><svg x="488.4" y="78.76" width="27.2" height="24.8" viewBox="0 0 56 51" fill="#FF5F46"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/></svg><rect x="530" y="71.0" width="40" height="40" rx="9.6" fill="#fff" stroke="none" stroke-width="0" filter="url(#sk-sh)"/><svg x="536.4" y="78.76" width="27.2" height="27.2" viewBox="0 0 56 51" fill="#FF5F46"><rect x="14" y="6" width="34" height="3"/>
  <rect x="14" y="6" width="3" height="33"/>
  <rect x="45" y="6" width="3" height="19"/>
  <rect x="14" y="17" width="34" height="3"/>
  <rect x="24" y="20" width="3" height="17"/>
  <rect x="14" y="37" width="17" height="3"/>
  <rect x="32.015" y="26.436" width="10.5" height="3.5" transform="rotate(131 37.265 28.186)"/>
  <rect x="32.5" y="31" width="15" height="3"/>
  <rect x="37.485" y="35.064" width="10.5" height="3.5" transform="rotate(131 42.735 36.814)"/></svg><text x="745.0" y="85.0" font-size="18" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Streaming Table</text><text x="745.0" y="107.0" font-size="14" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Materialized View</text><rect x="470" y="224" width="450" height="70" rx="12" fill="#2272B4" stroke="#0D47A1" stroke-width="2" filter="url(#sk-sh)"/><rect x="482" y="239.0" width="40" height="40" rx="9.6" fill="#fff" stroke="none" stroke-width="0" filter="url(#sk-sh)"/><svg x="488.4" y="245.4" width="27.2" height="27.2" viewBox="1.265 25.6 47.748 47.748" fill="none"><path d="M43.4295 61.3529L26.3179 31.8768C25.9325 31.2299 24.9818 31.2299 24.5964 31.8768L10.0541 56.9276C9.79712 57.3676 10.2596 57.8851 10.7221 57.6522C10.9019 57.5746 11.1332 57.4711 11.3644 57.3934C11.3644 57.3934 11.7498 57.2382 12.4435 57.0311C13.1886 56.7982 13.9851 56.5912 14.7302 56.41C16.7857 55.9442 19.4835 55.556 21.8986 55.9442C24.1339 56.3065 25.7012 56.9276 26.8317 57.5228C27.3456 57.8075 27.8338 57.1088 27.4227 56.6947C25.804 55.1161 23.2861 53.1752 19.4835 52.3729C19.3293 52.347 19.2265 52.14 19.3293 52.0106L25.0332 42.1508C25.213 41.8143 25.6755 41.8143 25.8811 42.1508L36.8264 60.9906C37.0576 61.3788 36.8521 61.8964 36.441 62.0516C35.5931 62.3363 34.5654 62.5433 33.4092 62.5433C32.6898 62.5433 31.9447 62.4657 31.1996 62.2845C31.1996 62.2845 29.0927 61.9481 26.2408 60.2401C23.4145 58.5321 17.9419 56.3583 11.8269 59.8778C9.20618 61.5599 7.45904 63.889 6.53408 65.3383C6.37992 65.5712 6.63686 65.83 6.8681 65.7006C8.97494 64.5619 14.6531 61.8964 19.7661 62.8539C22.7979 63.4232 26.2922 65.7264 29.8892 67.0721C35.8757 69.2977 41.7338 65.5712 41.7338 65.5453C42.3504 65.1312 42.7101 64.8207 42.7872 64.7689L42.8643 64.6913C43.8406 63.8632 44.0719 62.4657 43.4295 61.3529Z" fill="#00ADD4"/></svg><text x="721.0" y="253.0" font-size="18" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Sink</text><text x="721.0" y="275.0" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Delta · Kafka · Custom</text><path d="M212,165 C250,165 235,81 268,81" fill="none" stroke="#5A6F77" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/><rect x="270" y="64" width="150" height="34" rx="17" fill="#F5F5F5" stroke="#BDBDBD" stroke-width="1.4" /><text x="345" y="86" font-size="14" font-weight="600" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Default</text><path d="M422,81 H468" fill="none" stroke="#5A6F77" stroke-width="2.2" marker-end="url(#sk-ah-g)" stroke-linecap="round" stroke-linejoin="round"/><path d="M212,165 C250,165 235,249 268,249" fill="none" stroke="#5A6F77" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/><rect x="270" y="232" width="150" height="34" rx="17" fill="#F5F5F5" stroke="#BDBDBD" stroke-width="1.4" /><text x="345" y="254" font-size="14" font-weight="600" fill="#0B2026" text-anchor="middle" style="font-family:Consolas,Monaco,monospace" >@dp.append_flow</text><path d="M422,249 H468" fill="none" stroke="#5A6F77" stroke-width="2.2" marker-end="url(#sk-ah-g)" stroke-linecap="round" stroke-linejoin="round"/></svg>

<div style="background:#F8F9FA;border-left:4px solid #607D8B;padding:14px 18px;margin:24px 0;border-radius:4px;">
  <div style="font-size:16px;color:#444;line-height:1.8;">
    <strong>Only the Python API is supported</strong> — SQL is not supported for sinks. Only append flows (<code>append_flow</code> and <code>update_flow</code>) can write to a sink.
  </div>
</div>

**Reference Documentation:**
- [Sinks in SDP](https://docs.databricks.com/aws/en/ldp/sinks)


### B1. Supported Sink Types

Databricks supports <strong>several types of sinks</strong>. The four most common are shown here, each suited for a different destination and use case.


<div style="flex-wrap:wrap;display:flex;gap:0;margin:24px 0;align-items:stretch;">

  <div style="flex:1 1 150px;min-width:0;background:#E8F5E9;border-top:4px solid #388E3C;border-radius:12px 0 0 12px;padding:24px;">
    <div style="font-weight:bold;color:#1B5E20;font-size:16px;margin-bottom:14px;">Delta Table Sink</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Unity Catalog managed tables</li>
      <li>External Delta tables</li>
      <li>Write by path or table name</li>
    </ul>
  </div>

  <div style="flex:1 1 150px;min-width:0;background:#E3F2FD;border-top:4px solid #1976D2;padding:24px;">
    <div style="font-weight:bold;color:#0D47A1;font-size:16px;margin-bottom:14px;">Apache Kafka Sink</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Write back to Kafka topics</li>
      <li>Low-latency operational use cases</li>
      <li>Reverse ETL out of Databricks</li>
    </ul>
  </div>

  <div style="flex:1 1 150px;min-width:0;background:#FFF3E0;border-top:4px solid #F57C00;padding:24px;">
    <div style="font-weight:bold;color:#E65100;font-size:16px;margin-bottom:14px;">Azure Event Hubs Sink</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Uses Kafka interface format</li>
      <li>Real-time event streaming</li>
      <li>Fraud detection · recommendations</li>
    </ul>
  </div>

  <div style="flex:1 1 150px;min-width:0;background:#F3E5F5;border-top:4px solid #7B1FA2;border-radius:0 12px 12px 0;padding:24px;">
    <div style="font-weight:bold;color:#6A1B9A;font-size:16px;margin-bottom:14px;">Python Custom Sink</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Write to any data store</li>
      <li>Uses PySpark custom data sources</li>
      <li>Maximum flexibility</li>
    </ul>
  </div>

</div>

<div style="background:#E8F5E9;border-left:4px solid #388E3C;padding:14px 18px;margin:20px 0;border-radius:4px;">
  <div style="font-size:16px;color:#1B5E20;line-height:1.8;">
    For code examples of each sink type, refer to the <a href="https://docs.databricks.com/aws/en/ldp/ldp-sinks?language=Delta%C2%A0sinks#create-a-sink" style="color:#1976D2;">Creating a Sink Documentation</a>
  </div>
</div>


### B2. Managed Tables vs. Sinks

Every standard dataset in a Spark Declarative Pipeline — streaming table or materialized view — is **owned and managed by the pipeline**. A **sink** breaks this intentionally: it lets the pipeline write streaming data to a **plain Delta table that exists outside the pipeline's managed scope**.

<div style="flex-wrap:wrap;display:flex;gap:0;margin:24px 0;align-items:stretch;">

  <div style="flex:1 1 280px;min-width:0;background:#E8F5E9;border-top:4px solid #388E3C;border-radius:12px 0 0 12px;padding:24px;">
    <div style="font-weight:bold;color:#1B5E20;font-size:16px;margin-bottom:14px;">Managed Table (Default)</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Data stays within Unity Catalog</li>
      <li>Full pipeline lineage tracking</li>
      <li>Supports expectations and CDC</li>
      <li>Streaming Tables and Materialized Views</li>
    </ul>
  </div>

  <div style="display:flex;align-items:center;justify-content:center;padding:0 20px;background:#F8F9FA;border-top:1px solid #E0E0E0;border-bottom:1px solid #E0E0E0;">
    <div style="font-size:24px;color:#999;font-weight:300;">vs</div>
  </div>

  <div style="flex:1 1 280px;min-width:0;background:#E3F2FD;border-top:4px solid #1976D2;border-radius:0 12px 12px 0;padding:24px;">
    <div style="font-weight:bold;color:#0D47A1;font-size:16px;margin-bottom:14px;">Sink</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Write to external systems outside Databricks</li>
      <li>Enables reverse ETL and operational use cases</li>
      <li>Supports Kafka, Event Hubs, custom targets</li>
      <li>No expectations — append only</li>
    </ul>
  </div>
</div>


### B3. Delta Sink In Action

A <strong>Delta sink</strong> writes pipeline output to a Delta table <em>outside</em> the pipeline's managed lifecycle — unlocking configurations not possible on pipeline-managed streaming tables, such as <strong>Iceberg compatibility</strong>.

---

<div style="flex-wrap:wrap;display:flex;gap:16px;margin:16px 0 28px 0;">
  <div style="flex:1 1 280px;min-width:0;background:#FFF3E0;border-left:4px solid #F57C00;border-radius:4px;padding:16px 20px;">
    <div style="font-weight:bold;color:#E65100;font-size:16px;margin-bottom:8px;">Streaming Tables Cannot</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Enable Iceberg compatibility</li>
      <li>Share with non-Databricks platforms</li>
    </ul>
  </div>
  <div style="flex:1 1 280px;min-width:0;background:#E8F5E9;border-left:4px solid #388E3C;border-radius:4px;padding:16px 20px;">
    <div style="font-weight:bold;color:#1B5E20;font-size:16px;margin-bottom:8px;">Delta Sink Can</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Have full Delta table property control</li>
      <li>Enables Iceberg UniForm for cross-platform reads</li>
    </ul>
  </div>
</div>

---

#### Implementation

<div style="flex-wrap:wrap;display:flex;gap:0;margin:20px 0;align-items:stretch;">
  <div style="flex:1 1 511px;min-width:0;background:#E8F5E9;border-top:4px solid #388E3C;border-radius:12px 0 0 12px;padding:24px;">
    <div style="font-weight:bold;color:#1B5E20;font-size:16px;margin-bottom:16px;">Step 1 — Register the Sink</div>
    <div style="position:relative;"><button type="button" onclick="var b=this,t=document.createElement('textarea');t.value=b.parentNode.querySelector('pre').textContent.replace(/\u00a0/g,'');document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);b.textContent='✓ Copied!';setTimeout(function(){b.textContent='Copy';},2000);" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:14px;background:#fff;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;">Copy</button><pre style="background:#F8F9FA;border:1px solid #E0E0E0;border-radius:8px;padding:14px 16px;padding-right:90px;margin:0;font-size:15px;line-height:1.6;overflow-x:auto;white-space:pre;font-family:Consolas,Monaco,monospace;"><span style="color:#C92A18;font-weight:600;">from</span> pyspark <span style="color:#C92A18;font-weight:600;">import</span> pipelines <span style="color:#C92A18;font-weight:600;">as</span> dp
&nbsp;
dp.create_sink(
    name    = <span style="color:#0A7A4B;">&quot;my_sink&quot;</span>,
    format  = <span style="color:#0A7A4B;">&quot;delta&quot;</span>,
    options = {
        <span style="color:#0A7A4B;">&quot;tableName&quot;</span>: <span style="color:#0A7A4B;">&quot;catalog.schema.table&quot;</span>
    }
)</pre></div>
  </div>

  <div style="flex:1 1 435px;min-width:0;background:#E3F2FD;border-top:4px solid #1976D2;border-radius:0 12px 12px 0;padding:24px;">
    <div style="font-weight:bold;color:#0D47A1;font-size:16px;margin-bottom:16px;">Step 2 — Write to the Sink</div>
    <div style="position:relative;"><button type="button" onclick="var b=this,t=document.createElement('textarea');t.value=b.parentNode.querySelector('pre').textContent.replace(/\u00a0/g,'');document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);b.textContent='✓ Copied!';setTimeout(function(){b.textContent='Copy';},2000);" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:14px;background:#fff;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;">Copy</button><pre style="background:#F8F9FA;border:1px solid #E0E0E0;border-radius:8px;padding:14px 16px;padding-right:90px;margin:0;font-size:15px;line-height:1.6;overflow-x:auto;white-space:pre;font-family:Consolas,Monaco,monospace;">@dp.append_flow(
    name   = <span style="color:#0A7A4B;">&quot;my_sink_flow&quot;</span>,
    target = <span style="color:#0A7A4B;">&quot;my_sink&quot;</span>
)
<span style="color:#C92A18;font-weight:600;">def</span> my_sink_flow():
    <span style="color:#C92A18;font-weight:600;">return</span> spark.readStream.table(
        <span style="color:#0A7A4B;">&quot;schema.source_table&quot;</span>
    )</pre></div>
  </div>
</div>

<div style="background:#F8F9FA;border-left:4px solid #607D8B;border-radius:4px;padding:16px 20px;margin-top:8px;">
  <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
    <li>Checkpointing is handled automatically by <code>append_flow</code></li>
    <li>Only new records written per run — no overwrites</li>
    <li>Python only — no SQL equivalent</li>
  </ul>
</div>

## C. Iceberg Reads via Delta UniForm

Delta UniForm enables cross-platform access to Delta tables by automatically generating Apache Iceberg metadata without duplicating the underlying data.

### C1. The Cross-Platform Challenge

<div style="font-size:16px;color:#555;line-height:1.8;margin-bottom:24px;">
  Modern data architectures span multiple platforms. Traditionally, supporting each platform meant <strong>copying data into multiple formats</strong> — creating duplication, sync issues, and increased storage costs. <strong>Delta UniForm</strong> solves this with a single set of files and two metadata layers.
</div>

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 350" width="100%" role="img" aria-label="Traditional approach copies data into several formats; Delta UniForm keeps one set of Parquet files with Delta and Iceberg metadata" style="font-family:'DM Sans',Arial,Helvetica,sans-serif;max-width:960px;display:block;margin:12px auto;"><defs><filter id="uc-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter><marker id="uc-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#5A6F77"/></marker><marker id="uc-ah-l" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF3621"/></marker><marker id="uc-ah-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#2272B4"/></marker><marker id="uc-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="uc-ah-k" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker></defs><rect x="10" y="10" width="450" height="330" rx="16" fill="#FFF3F3" stroke="#C62828" stroke-width="2" /><text x="28" y="40" font-size="17" font-weight="700" fill="#C62828" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Traditional Approach</text><rect x="540" y="10" width="410" height="330" rx="16" fill="#F1F8E9" stroke="#388E3C" stroke-width="2" /><text x="558" y="40" font-size="17" font-weight="700" fill="#388E3C" text-anchor="start" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Delta UniForm</text><rect x="150" y="62" width="170" height="50" rx="12" fill="#FFEBEE" stroke="#C62828" stroke-width="2" filter="url(#uc-sh)"/><rect x="162" y="72.0" width="30" height="30" rx="7.199999999999999" fill="#fff" stroke="none" stroke-width="0" filter="url(#uc-sh)"/><svg x="166.8" y="76.8" width="20.4" height="20.4" viewBox="1.265 25.6 47.748 47.748" fill="none"><path d="M43.4295 61.3529L26.3179 31.8768C25.9325 31.2299 24.9818 31.2299 24.5964 31.8768L10.0541 56.9276C9.79712 57.3676 10.2596 57.8851 10.7221 57.6522C10.9019 57.5746 11.1332 57.4711 11.3644 57.3934C11.3644 57.3934 11.7498 57.2382 12.4435 57.0311C13.1886 56.7982 13.9851 56.5912 14.7302 56.41C16.7857 55.9442 19.4835 55.556 21.8986 55.9442C24.1339 56.3065 25.7012 56.9276 26.8317 57.5228C27.3456 57.8075 27.8338 57.1088 27.4227 56.6947C25.804 55.1161 23.2861 53.1752 19.4835 52.3729C19.3293 52.347 19.2265 52.14 19.3293 52.0106L25.0332 42.1508C25.213 41.8143 25.6755 41.8143 25.8811 42.1508L36.8264 60.9906C37.0576 61.3788 36.8521 61.8964 36.441 62.0516C35.5931 62.3363 34.5654 62.5433 33.4092 62.5433C32.6898 62.5433 31.9447 62.4657 31.1996 62.2845C31.1996 62.2845 29.0927 61.9481 26.2408 60.2401C23.4145 58.5321 17.9419 56.3583 11.8269 59.8778C9.20618 61.5599 7.45904 63.889 6.53408 65.3383C6.37992 65.5712 6.63686 65.83 6.8681 65.7006C8.97494 64.5619 14.6531 61.8964 19.7661 62.8539C22.7979 63.4232 26.2922 65.7264 29.8892 67.0721C35.8757 69.2977 41.7338 65.5712 41.7338 65.5453C42.3504 65.1312 42.7101 64.8207 42.7872 64.7689L42.8643 64.6913C43.8406 63.8632 44.0719 62.4657 43.4295 61.3529Z" fill="#00ADD4"/></svg><text x="256.0" y="92.0" font-size="16" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Delta Files</text><rect x="150" y="150" width="170" height="50" rx="12" fill="#FFCDD2" stroke="#C62828" stroke-width="2" filter="url(#uc-sh)"/><text x="235.0" y="180.0" font-size="16" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Copy and Convert</text><path d="M235,114 V146" fill="none" stroke="#C62828" stroke-width="2.4" marker-end="url(#uc-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="18" y="254" width="136" height="50" rx="12" fill="#FFEBEE" stroke="#C62828" stroke-width="2" filter="url(#uc-sh)"/><rect x="30" y="266.0" width="26" height="26" rx="6.24" fill="#fff" stroke="none" stroke-width="0" filter="url(#uc-sh)"/><svg x="34.16" y="270.16" width="17.68" height="17.68" viewBox="579.592 -3.681 222.861 222.861" fill="none"><path style="opacity:0.973" fill="#b8e1f3" d="M 764.5,63.5 C 750.833,63.5 737.167,63.5 723.5,63.5C 723.5,61.5 723.5,59.5 723.5,57.5C 728.607,57.8345 732.94,57.5012 736.5,56.5C 734.073,54.4424 731.407,52.6091 728.5,51C 706.167,50.3333 683.833,50.3333 661.5,51C 657.764,52.4098 654.098,53.9098 650.5,55.5C 647.053,57.0541 645.053,59.7208 644.5,63.5C 659.833,63.5 675.167,63.5 690.5,63.5C 690.173,73.6806 690.506,83.6806 691.5,93.5C 691.5,97.8333 691.5,102.167 691.5,106.5C 691.167,106.5 690.833,106.5 690.5,106.5C 672.032,98.7651 653.699,90.7651 635.5,82.5C 648.016,76.4095 660.683,70.5762 673.5,65C 663.506,64.5002 653.506,64.3335 643.5,64.5C 635.016,63.507 626.349,63.1736 617.5,63.5C 634.269,36.5202 658.769,22.8535 691,22.5C 723.231,22.8535 747.731,36.5202 764.5,63.5 Z"/>
<path style="opacity:1" fill="#f5fafd" d="M 736.5,56.5 C 707.662,56.8313 678.995,56.498 650.5,55.5C 654.098,53.9098 657.764,52.4098 661.5,51C 683.833,50.3333 706.167,50.3333 728.5,51C 731.407,52.6091 734.073,54.4424 736.5,56.5 Z"/>
<path style="opacity:1" fill="#eaf7fd" d="M 650.5,55.5 C 678.995,56.498 707.662,56.8313 736.5,56.5C 732.94,57.5012 728.607,57.8345 723.5,57.5C 723.5,59.5 723.5,61.5 723.5,63.5C 712.5,63.5 701.5,63.5 690.5,63.5C 675.167,63.5 659.833,63.5 644.5,63.5C 645.053,59.7208 647.053,57.0541 650.5,55.5 Z"/>
<path style="opacity:1" fill="#4b94cb" d="M 617.5,63.5 C 626.349,63.1736 635.016,63.507 643.5,64.5C 643.5,65.1667 643.167,65.5 642.5,65.5C 634.167,64.8333 625.833,64.1667 617.5,63.5 Z"/>
<path style="opacity:0.985" fill="#277abe" d="M 617.5,63.5 C 625.833,64.1667 634.167,64.8333 642.5,65.5C 640.013,70.4734 637.513,75.4734 635,80.5C 636.926,89.255 640.26,97.5884 645,105.5C 650.7,114.257 656.867,122.591 663.5,130.5C 663.569,135.854 663.902,141.187 664.5,146.5C 664.351,147.552 664.517,148.552 665,149.5C 673.707,158.704 682.207,168.037 690.5,177.5C 691.077,178.773 691.743,178.773 692.5,177.5C 699.178,166.139 705.845,154.806 712.5,143.5C 717.46,139.712 722.293,135.712 727,131.5C 727.5,122.506 727.666,113.506 727.5,104.5C 733.833,97.1667 740.167,89.8333 746.5,82.5C 746.804,81.8501 747.137,81.1835 747.5,80.5C 745.5,75.5 743.5,70.5 741.5,65.5C 749.472,64.5023 757.472,64.169 765.5,64.5C 777.293,84.3207 780.793,105.654 776,128.5C 767.539,160.962 747.372,182.462 715.5,193C 672.603,202.376 639.103,188.876 615,152.5C 602.206,127.892 600.539,102.559 610,76.5C 612.022,71.7768 614.522,67.4434 617.5,63.5 Z"/>
<path style="opacity:1" fill="#68c1ea" d="M 723.5,64.5 C 723.5,73.8333 723.5,83.1667 723.5,92.5C 713.663,97.3434 703.663,102.01 693.5,106.5C 692.833,106.5 692.5,106.833 692.5,107.5C 692.167,107.5 691.833,107.5 691.5,107.5C 691.5,107.167 691.5,106.833 691.5,106.5C 691.5,102.167 691.5,97.8333 691.5,93.5C 691.5,83.8333 691.5,74.1667 691.5,64.5C 702.167,64.5 712.833,64.5 723.5,64.5 Z"/>
<path style="opacity:1" fill="#4e8bc9" d="M 723.5,63.5 C 737.167,63.5 750.833,63.5 764.5,63.5C 765.167,63.5 765.5,63.8333 765.5,64.5C 757.472,64.169 749.472,64.5023 741.5,65.5C 743.5,70.5 745.5,75.5 747.5,80.5C 747.137,81.1835 746.804,81.8501 746.5,82.5C 738.712,85.5608 731.045,88.8941 723.5,92.5C 723.5,83.1667 723.5,73.8333 723.5,64.5C 723.5,64.1667 723.5,63.8333 723.5,63.5 Z"/>
<path style="opacity:1" fill="#a2d8f1" d="M 690.5,63.5 C 701.5,63.5 712.5,63.5 723.5,63.5C 723.5,63.8333 723.5,64.1667 723.5,64.5C 712.833,64.5 702.167,64.5 691.5,64.5C 691.5,74.1667 691.5,83.8333 691.5,93.5C 690.506,83.6806 690.173,73.6806 690.5,63.5 Z"/>
<path style="opacity:1" fill="#90d4f0" d="M 643.5,64.5 C 653.506,64.3335 663.506,64.5002 673.5,65C 660.683,70.5762 648.016,76.4095 635.5,82.5C 653.699,90.7651 672.032,98.7651 690.5,106.5C 690.141,108.371 689.141,109.871 687.5,111C 679.021,116.985 671.021,123.485 663.5,130.5C 656.867,122.591 650.7,114.257 645,105.5C 640.26,97.5884 636.926,89.255 635,80.5C 637.513,75.4734 640.013,70.4734 642.5,65.5C 643.167,65.5 643.5,65.1667 643.5,64.5 Z"/>
<path style="opacity:1" fill="#266fa8" d="M 746.5,82.5 C 740.167,89.8333 733.833,97.1667 727.5,104.5C 716.812,105.28 706.145,106.28 695.5,107.5C 694.584,107.457 693.917,107.124 693.5,106.5C 703.663,102.01 713.663,97.3434 723.5,92.5C 731.045,88.8941 738.712,85.5608 746.5,82.5 Z"/>
<path style="opacity:1" fill="#4d8ac8" d="M 727.5,104.5 C 727.666,113.506 727.5,122.506 727,131.5C 722.293,135.712 717.46,139.712 712.5,143.5C 712.167,142.833 711.833,142.167 711.5,141.5C 704.628,130.419 698.295,119.086 692.5,107.5C 692.5,106.833 692.833,106.5 693.5,106.5C 693.917,107.124 694.584,107.457 695.5,107.5C 706.145,106.28 716.812,105.28 727.5,104.5 Z"/>
<path style="opacity:1" fill="#abdcf1" d="M 690.5,106.5 C 690.833,106.5 691.167,106.5 691.5,106.5C 691.5,106.833 691.5,107.167 691.5,107.5C 691.5,115.5 691.5,123.5 691.5,131.5C 691.167,131.5 690.833,131.5 690.5,131.5C 681.502,131.119 672.502,130.786 663.5,130.5C 671.021,123.485 679.021,116.985 687.5,111C 689.141,109.871 690.141,108.371 690.5,106.5 Z"/>
<path style="opacity:1" fill="#54a5d7" d="M 691.5,107.5 C 691.833,107.5 692.167,107.5 692.5,107.5C 698.295,119.086 704.628,130.419 711.5,141.5C 704.456,138.721 697.789,135.388 691.5,131.5C 691.5,123.5 691.5,115.5 691.5,107.5 Z"/>
<path style="opacity:1" fill="#b6e0f2" d="M 663.5,130.5 C 672.502,130.786 681.502,131.119 690.5,131.5C 690.5,138.833 690.5,146.167 690.5,153.5C 683.125,152.66 675.791,151.493 668.5,150C 666.973,148.968 665.64,147.801 664.5,146.5C 663.902,141.187 663.569,135.854 663.5,130.5 Z"/>
<path style="opacity:1" fill="#58afde" d="M 690.5,131.5 C 690.833,131.5 691.167,131.5 691.5,131.5C 697.789,135.388 704.456,138.721 711.5,141.5C 711.833,142.167 712.167,142.833 712.5,143.5C 705.734,146.382 699.067,149.549 692.5,153C 691.544,156.394 691.211,159.894 691.5,163.5C 690.521,160.375 690.187,157.042 690.5,153.5C 690.5,146.167 690.5,138.833 690.5,131.5 Z"/>
<path style="opacity:1" fill="#266fa8" d="M 712.5,143.5 C 705.845,154.806 699.178,166.139 692.5,177.5C 691.743,178.773 691.077,178.773 690.5,177.5C 691.486,173.03 691.819,168.363 691.5,163.5C 691.211,159.894 691.544,156.394 692.5,153C 699.067,149.549 705.734,146.382 712.5,143.5 Z"/>
<path style="opacity:1" fill="#8fd3f0" d="M 664.5,146.5 C 665.64,147.801 666.973,148.968 668.5,150C 675.791,151.493 683.125,152.66 690.5,153.5C 690.187,157.042 690.521,160.375 691.5,163.5C 691.819,168.363 691.486,173.03 690.5,177.5C 682.207,168.037 673.707,158.704 665,149.5C 664.517,148.552 664.351,147.552 664.5,146.5 Z"/></svg><text x="105.0" y="284.0" font-size="14" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Iceberg Copy</text><path d="M235,202 C235,230 86,226 86,250" fill="none" stroke="#C62828" stroke-width="2.2" marker-end="url(#uc-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="163" y="254" width="136" height="50" rx="12" fill="#FFEBEE" stroke="#C62828" stroke-width="2" filter="url(#uc-sh)"/><text x="231.0" y="284.0" font-size="14" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Parquet Copy</text><path d="M235,202 C235,230 231,226 231,250" fill="none" stroke="#C62828" stroke-width="2.2" marker-end="url(#uc-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="308" y="254" width="136" height="50" rx="12" fill="#FFEBEE" stroke="#C62828" stroke-width="2" filter="url(#uc-sh)"/><text x="376.0" y="284.0" font-size="14" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Hive Copy</text><path d="M235,202 C235,230 376,226 376,250" fill="none" stroke="#C62828" stroke-width="2.2" marker-end="url(#uc-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><rect x="650" y="56" width="224" height="64" rx="12" fill="#E8F5E9" stroke="#388E3C" stroke-width="2" filter="url(#uc-sh)"/><rect x="662" y="68.0" width="40" height="40" rx="9.6" fill="#fff" stroke="none" stroke-width="0" filter="url(#uc-sh)"/><g transform="translate(671.664,73.85600000000001) scale(0.591304347826087)"><path d="M0,0 h22 l12,12 v32 h-34 z" fill="#fff" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><path d="M22,0 v12 h12" fill="none" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><rect x="2" y="25" width="30" height="14" rx="3" fill="#FF5F46"/><text x="17" y="35.5" font-size="9.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,sans-serif">PQ</text></g><text x="788.0" y="82.0" font-size="15.5" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >One Set of</text><text x="788.0" y="104.0" font-size="15.5" font-weight="400" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Parquet Files</text><rect x="552" y="230" width="192" height="56" rx="12" fill="#E6F0F8" stroke="#2272B4" stroke-width="2" filter="url(#uc-sh)"/><rect x="564" y="242.0" width="32" height="32" rx="7.68" fill="#fff" stroke="none" stroke-width="0" filter="url(#uc-sh)"/><svg x="569.12" y="247.12" width="21.759999999999998" height="21.759999999999998" viewBox="1.265 25.6 47.748 47.748" fill="none"><path d="M43.4295 61.3529L26.3179 31.8768C25.9325 31.2299 24.9818 31.2299 24.5964 31.8768L10.0541 56.9276C9.79712 57.3676 10.2596 57.8851 10.7221 57.6522C10.9019 57.5746 11.1332 57.4711 11.3644 57.3934C11.3644 57.3934 11.7498 57.2382 12.4435 57.0311C13.1886 56.7982 13.9851 56.5912 14.7302 56.41C16.7857 55.9442 19.4835 55.556 21.8986 55.9442C24.1339 56.3065 25.7012 56.9276 26.8317 57.5228C27.3456 57.8075 27.8338 57.1088 27.4227 56.6947C25.804 55.1161 23.2861 53.1752 19.4835 52.3729C19.3293 52.347 19.2265 52.14 19.3293 52.0106L25.0332 42.1508C25.213 41.8143 25.6755 41.8143 25.8811 42.1508L36.8264 60.9906C37.0576 61.3788 36.8521 61.8964 36.441 62.0516C35.5931 62.3363 34.5654 62.5433 33.4092 62.5433C32.6898 62.5433 31.9447 62.4657 31.1996 62.2845C31.1996 62.2845 29.0927 61.9481 26.2408 60.2401C23.4145 58.5321 17.9419 56.3583 11.8269 59.8778C9.20618 61.5599 7.45904 63.889 6.53408 65.3383C6.37992 65.5712 6.63686 65.83 6.8681 65.7006C8.97494 64.5619 14.6531 61.8964 19.7661 62.8539C22.7979 63.4232 26.2922 65.7264 29.8892 67.0721C35.8757 69.2977 41.7338 65.5712 41.7338 65.5453C42.3504 65.1312 42.7101 64.8207 42.7872 64.7689L42.8643 64.6913C43.8406 63.8632 44.0719 62.4657 43.4295 61.3529Z" fill="#00ADD4"/></svg><text x="670.0" y="263.0" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Delta Metadata</text><path d="M762,122 C762,170 648,180 648,226" fill="none" stroke="#00A972" stroke-width="2.4" marker-end="url(#uc-ah-k)" stroke-linecap="round" stroke-linejoin="round"/><rect x="752" y="230" width="192" height="56" rx="12" fill="#E6F0F8" stroke="#2272B4" stroke-width="2" filter="url(#uc-sh)"/><rect x="764" y="242.0" width="32" height="32" rx="7.68" fill="#fff" stroke="none" stroke-width="0" filter="url(#uc-sh)"/><svg x="769.12" y="247.12" width="21.759999999999998" height="21.759999999999998" viewBox="579.592 -3.681 222.861 222.861" fill="none"><path style="opacity:0.973" fill="#b8e1f3" d="M 764.5,63.5 C 750.833,63.5 737.167,63.5 723.5,63.5C 723.5,61.5 723.5,59.5 723.5,57.5C 728.607,57.8345 732.94,57.5012 736.5,56.5C 734.073,54.4424 731.407,52.6091 728.5,51C 706.167,50.3333 683.833,50.3333 661.5,51C 657.764,52.4098 654.098,53.9098 650.5,55.5C 647.053,57.0541 645.053,59.7208 644.5,63.5C 659.833,63.5 675.167,63.5 690.5,63.5C 690.173,73.6806 690.506,83.6806 691.5,93.5C 691.5,97.8333 691.5,102.167 691.5,106.5C 691.167,106.5 690.833,106.5 690.5,106.5C 672.032,98.7651 653.699,90.7651 635.5,82.5C 648.016,76.4095 660.683,70.5762 673.5,65C 663.506,64.5002 653.506,64.3335 643.5,64.5C 635.016,63.507 626.349,63.1736 617.5,63.5C 634.269,36.5202 658.769,22.8535 691,22.5C 723.231,22.8535 747.731,36.5202 764.5,63.5 Z"/>
<path style="opacity:1" fill="#f5fafd" d="M 736.5,56.5 C 707.662,56.8313 678.995,56.498 650.5,55.5C 654.098,53.9098 657.764,52.4098 661.5,51C 683.833,50.3333 706.167,50.3333 728.5,51C 731.407,52.6091 734.073,54.4424 736.5,56.5 Z"/>
<path style="opacity:1" fill="#eaf7fd" d="M 650.5,55.5 C 678.995,56.498 707.662,56.8313 736.5,56.5C 732.94,57.5012 728.607,57.8345 723.5,57.5C 723.5,59.5 723.5,61.5 723.5,63.5C 712.5,63.5 701.5,63.5 690.5,63.5C 675.167,63.5 659.833,63.5 644.5,63.5C 645.053,59.7208 647.053,57.0541 650.5,55.5 Z"/>
<path style="opacity:1" fill="#4b94cb" d="M 617.5,63.5 C 626.349,63.1736 635.016,63.507 643.5,64.5C 643.5,65.1667 643.167,65.5 642.5,65.5C 634.167,64.8333 625.833,64.1667 617.5,63.5 Z"/>
<path style="opacity:0.985" fill="#277abe" d="M 617.5,63.5 C 625.833,64.1667 634.167,64.8333 642.5,65.5C 640.013,70.4734 637.513,75.4734 635,80.5C 636.926,89.255 640.26,97.5884 645,105.5C 650.7,114.257 656.867,122.591 663.5,130.5C 663.569,135.854 663.902,141.187 664.5,146.5C 664.351,147.552 664.517,148.552 665,149.5C 673.707,158.704 682.207,168.037 690.5,177.5C 691.077,178.773 691.743,178.773 692.5,177.5C 699.178,166.139 705.845,154.806 712.5,143.5C 717.46,139.712 722.293,135.712 727,131.5C 727.5,122.506 727.666,113.506 727.5,104.5C 733.833,97.1667 740.167,89.8333 746.5,82.5C 746.804,81.8501 747.137,81.1835 747.5,80.5C 745.5,75.5 743.5,70.5 741.5,65.5C 749.472,64.5023 757.472,64.169 765.5,64.5C 777.293,84.3207 780.793,105.654 776,128.5C 767.539,160.962 747.372,182.462 715.5,193C 672.603,202.376 639.103,188.876 615,152.5C 602.206,127.892 600.539,102.559 610,76.5C 612.022,71.7768 614.522,67.4434 617.5,63.5 Z"/>
<path style="opacity:1" fill="#68c1ea" d="M 723.5,64.5 C 723.5,73.8333 723.5,83.1667 723.5,92.5C 713.663,97.3434 703.663,102.01 693.5,106.5C 692.833,106.5 692.5,106.833 692.5,107.5C 692.167,107.5 691.833,107.5 691.5,107.5C 691.5,107.167 691.5,106.833 691.5,106.5C 691.5,102.167 691.5,97.8333 691.5,93.5C 691.5,83.8333 691.5,74.1667 691.5,64.5C 702.167,64.5 712.833,64.5 723.5,64.5 Z"/>
<path style="opacity:1" fill="#4e8bc9" d="M 723.5,63.5 C 737.167,63.5 750.833,63.5 764.5,63.5C 765.167,63.5 765.5,63.8333 765.5,64.5C 757.472,64.169 749.472,64.5023 741.5,65.5C 743.5,70.5 745.5,75.5 747.5,80.5C 747.137,81.1835 746.804,81.8501 746.5,82.5C 738.712,85.5608 731.045,88.8941 723.5,92.5C 723.5,83.1667 723.5,73.8333 723.5,64.5C 723.5,64.1667 723.5,63.8333 723.5,63.5 Z"/>
<path style="opacity:1" fill="#a2d8f1" d="M 690.5,63.5 C 701.5,63.5 712.5,63.5 723.5,63.5C 723.5,63.8333 723.5,64.1667 723.5,64.5C 712.833,64.5 702.167,64.5 691.5,64.5C 691.5,74.1667 691.5,83.8333 691.5,93.5C 690.506,83.6806 690.173,73.6806 690.5,63.5 Z"/>
<path style="opacity:1" fill="#90d4f0" d="M 643.5,64.5 C 653.506,64.3335 663.506,64.5002 673.5,65C 660.683,70.5762 648.016,76.4095 635.5,82.5C 653.699,90.7651 672.032,98.7651 690.5,106.5C 690.141,108.371 689.141,109.871 687.5,111C 679.021,116.985 671.021,123.485 663.5,130.5C 656.867,122.591 650.7,114.257 645,105.5C 640.26,97.5884 636.926,89.255 635,80.5C 637.513,75.4734 640.013,70.4734 642.5,65.5C 643.167,65.5 643.5,65.1667 643.5,64.5 Z"/>
<path style="opacity:1" fill="#266fa8" d="M 746.5,82.5 C 740.167,89.8333 733.833,97.1667 727.5,104.5C 716.812,105.28 706.145,106.28 695.5,107.5C 694.584,107.457 693.917,107.124 693.5,106.5C 703.663,102.01 713.663,97.3434 723.5,92.5C 731.045,88.8941 738.712,85.5608 746.5,82.5 Z"/>
<path style="opacity:1" fill="#4d8ac8" d="M 727.5,104.5 C 727.666,113.506 727.5,122.506 727,131.5C 722.293,135.712 717.46,139.712 712.5,143.5C 712.167,142.833 711.833,142.167 711.5,141.5C 704.628,130.419 698.295,119.086 692.5,107.5C 692.5,106.833 692.833,106.5 693.5,106.5C 693.917,107.124 694.584,107.457 695.5,107.5C 706.145,106.28 716.812,105.28 727.5,104.5 Z"/>
<path style="opacity:1" fill="#abdcf1" d="M 690.5,106.5 C 690.833,106.5 691.167,106.5 691.5,106.5C 691.5,106.833 691.5,107.167 691.5,107.5C 691.5,115.5 691.5,123.5 691.5,131.5C 691.167,131.5 690.833,131.5 690.5,131.5C 681.502,131.119 672.502,130.786 663.5,130.5C 671.021,123.485 679.021,116.985 687.5,111C 689.141,109.871 690.141,108.371 690.5,106.5 Z"/>
<path style="opacity:1" fill="#54a5d7" d="M 691.5,107.5 C 691.833,107.5 692.167,107.5 692.5,107.5C 698.295,119.086 704.628,130.419 711.5,141.5C 704.456,138.721 697.789,135.388 691.5,131.5C 691.5,123.5 691.5,115.5 691.5,107.5 Z"/>
<path style="opacity:1" fill="#b6e0f2" d="M 663.5,130.5 C 672.502,130.786 681.502,131.119 690.5,131.5C 690.5,138.833 690.5,146.167 690.5,153.5C 683.125,152.66 675.791,151.493 668.5,150C 666.973,148.968 665.64,147.801 664.5,146.5C 663.902,141.187 663.569,135.854 663.5,130.5 Z"/>
<path style="opacity:1" fill="#58afde" d="M 690.5,131.5 C 690.833,131.5 691.167,131.5 691.5,131.5C 697.789,135.388 704.456,138.721 711.5,141.5C 711.833,142.167 712.167,142.833 712.5,143.5C 705.734,146.382 699.067,149.549 692.5,153C 691.544,156.394 691.211,159.894 691.5,163.5C 690.521,160.375 690.187,157.042 690.5,153.5C 690.5,146.167 690.5,138.833 690.5,131.5 Z"/>
<path style="opacity:1" fill="#266fa8" d="M 712.5,143.5 C 705.845,154.806 699.178,166.139 692.5,177.5C 691.743,178.773 691.077,178.773 690.5,177.5C 691.486,173.03 691.819,168.363 691.5,163.5C 691.211,159.894 691.544,156.394 692.5,153C 699.067,149.549 705.734,146.382 712.5,143.5 Z"/>
<path style="opacity:1" fill="#8fd3f0" d="M 664.5,146.5 C 665.64,147.801 666.973,148.968 668.5,150C 675.791,151.493 683.125,152.66 690.5,153.5C 690.187,157.042 690.521,160.375 691.5,163.5C 691.819,168.363 691.486,173.03 690.5,177.5C 682.207,168.037 673.707,158.704 665,149.5C 664.517,148.552 664.351,147.552 664.5,146.5 Z"/></svg><text x="870.0" y="263.0" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Iceberg Metadata</text><path d="M762,122 C762,170 848,180 848,226" fill="none" stroke="#00A972" stroke-width="2.4" marker-end="url(#uc-ah-k)" stroke-linecap="round" stroke-linejoin="round"/><text x="762" y="322" font-size="13.5" font-weight="400" fill="#388E3C" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >no copies — two metadata layers</text></svg>

<div style="flex-wrap:wrap;display:flex;gap:16px;margin:24px 0;">
  <div style="flex:1 1 280px;min-width:0;background:#FFEBEE;border-left:4px solid #C62828;border-radius:4px;padding:16px 20px;">
    <div style="font-weight:bold;color:#C62828;font-size:16px;margin-bottom:8px;">Traditional Approach</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Data copied into multiple formats</li>
      <li>Sync issues across copies</li>
      <li>Increased storage costs</li>
    </ul>
  </div>
  <div style="flex:1 1 280px;min-width:0;background:#E8F5E9;border-left:4px solid #388E3C;border-radius:4px;padding:16px 20px;">
    <div style="font-weight:bold;color:#1B5E20;font-size:16px;margin-bottom:8px;">Delta UniForm</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>One set of Parquet files</li>
      <li>Delta + Iceberg metadata layers</li>
      <li>No duplication — automatic sync</li>
    </ul>
  </div>
</div>

<div style="background:#E3F2FD;border-left:4px solid #1976D2;padding:14px 18px;border-radius:4px;">
  <div style="font-size:16px;color:#0D47A1;line-height:1.8;">
    For full details refer to the <a href="https://docs.databricks.com/aws/en/delta/uniform" style="color:#1976D2;font-weight:bold;">Delta UniForm Documentation →</a>
  </div>
</div>

### C2. How Delta UniForm Works

  Delta UniForm maintains <strong>one physical dataset</strong> with <strong>two logical views</strong> — no data duplication, no separate copies. Iceberg metadata is generated <strong>asynchronously</strong> after every Delta write, keeping both views in sync automatically.


---
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 390" width="100%" role="img" aria-label="One Delta table has one set of Parquet files with a Delta transaction log read by Databricks clients and Iceberg metadata read by external tools" style="font-family:'DM Sans',Arial,Helvetica,sans-serif;max-width:960px;display:block;margin:12px auto;"><defs><filter id="uh-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter><marker id="uh-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#5A6F77"/></marker><marker id="uh-ah-l" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF3621"/></marker><marker id="uh-ah-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#2272B4"/></marker><marker id="uh-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="uh-ah-k" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker></defs><rect x="10" y="20" width="570" height="350" rx="16" fill="#F8F9FA" stroke="#607D8B" stroke-width="2" /><text x="295" y="50" font-size="16" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >One Delta Table — One Set of Parquet Files</text><rect x="36" y="138" width="190" height="130" rx="14" fill="#fff" stroke="#8C98A0" stroke-width="1.8" filter="url(#uh-sh)"/><g transform="translate(52,160) scale(1.0)"><path d="M0,0 h22 l12,12 v32 h-34 z" fill="#fff" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><path d="M22,0 v12 h12" fill="none" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><rect x="2" y="25" width="30" height="14" rx="3" fill="#FF5F46"/><text x="17" y="35.5" font-size="9.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,sans-serif">PQ</text></g><g transform="translate(106,160) scale(1.0)"><path d="M0,0 h22 l12,12 v32 h-34 z" fill="#fff" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><path d="M22,0 v12 h12" fill="none" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><rect x="2" y="25" width="30" height="14" rx="3" fill="#FF5F46"/><text x="17" y="35.5" font-size="9.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,sans-serif">PQ</text></g><g transform="translate(160,160) scale(1.0)"><path d="M0,0 h22 l12,12 v32 h-34 z" fill="#fff" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><path d="M22,0 v12 h12" fill="none" stroke="#FF5F46" stroke-width="2" stroke-linejoin="round"/><rect x="2" y="25" width="30" height="14" rx="3" fill="#FF5F46"/><text x="17" y="35.5" font-size="9.5" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,sans-serif">PQ</text></g><text x="131" y="252" font-size="14.5" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Parquet Data Files</text><rect x="318" y="82" width="254" height="78" rx="12" fill="#2272B4" stroke="#0D47A1" stroke-width="2" filter="url(#uh-sh)"/><rect x="330" y="101.0" width="40" height="40" rx="9.6" fill="#fff" stroke="none" stroke-width="0" filter="url(#uh-sh)"/><svg x="336.4" y="107.4" width="27.2" height="27.2" viewBox="1.265 25.6 47.748 47.748" fill="none"><path d="M43.4295 61.3529L26.3179 31.8768C25.9325 31.2299 24.9818 31.2299 24.5964 31.8768L10.0541 56.9276C9.79712 57.3676 10.2596 57.8851 10.7221 57.6522C10.9019 57.5746 11.1332 57.4711 11.3644 57.3934C11.3644 57.3934 11.7498 57.2382 12.4435 57.0311C13.1886 56.7982 13.9851 56.5912 14.7302 56.41C16.7857 55.9442 19.4835 55.556 21.8986 55.9442C24.1339 56.3065 25.7012 56.9276 26.8317 57.5228C27.3456 57.8075 27.8338 57.1088 27.4227 56.6947C25.804 55.1161 23.2861 53.1752 19.4835 52.3729C19.3293 52.347 19.2265 52.14 19.3293 52.0106L25.0332 42.1508C25.213 41.8143 25.6755 41.8143 25.8811 42.1508L36.8264 60.9906C37.0576 61.3788 36.8521 61.8964 36.441 62.0516C35.5931 62.3363 34.5654 62.5433 33.4092 62.5433C32.6898 62.5433 31.9447 62.4657 31.1996 62.2845C31.1996 62.2845 29.0927 61.9481 26.2408 60.2401C23.4145 58.5321 17.9419 56.3583 11.8269 59.8778C9.20618 61.5599 7.45904 63.889 6.53408 65.3383C6.37992 65.5712 6.63686 65.83 6.8681 65.7006C8.97494 64.5619 14.6531 61.8964 19.7661 62.8539C22.7979 63.4232 26.2922 65.7264 29.8892 67.0721C35.8757 69.2977 41.7338 65.5712 41.7338 65.5453C42.3504 65.1312 42.7101 64.8207 42.7872 64.7689L42.8643 64.6913C43.8406 63.8632 44.0719 62.4657 43.4295 61.3529Z" fill="#00ADD4"/></svg><text x="471.0" y="115.0" font-size="15" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Delta Transaction Log</text><text x="471.0" y="137.0" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:Consolas,Monaco,monospace" >_delta_log/</text><rect x="318" y="250" width="254" height="78" rx="12" fill="#2E7D32" stroke="#1B5E20" stroke-width="2" filter="url(#uh-sh)"/><rect x="330" y="269.0" width="40" height="40" rx="9.6" fill="#fff" stroke="none" stroke-width="0" filter="url(#uh-sh)"/><svg x="336.4" y="275.4" width="27.2" height="27.2" viewBox="579.592 -3.681 222.861 222.861" fill="none"><path style="opacity:0.973" fill="#b8e1f3" d="M 764.5,63.5 C 750.833,63.5 737.167,63.5 723.5,63.5C 723.5,61.5 723.5,59.5 723.5,57.5C 728.607,57.8345 732.94,57.5012 736.5,56.5C 734.073,54.4424 731.407,52.6091 728.5,51C 706.167,50.3333 683.833,50.3333 661.5,51C 657.764,52.4098 654.098,53.9098 650.5,55.5C 647.053,57.0541 645.053,59.7208 644.5,63.5C 659.833,63.5 675.167,63.5 690.5,63.5C 690.173,73.6806 690.506,83.6806 691.5,93.5C 691.5,97.8333 691.5,102.167 691.5,106.5C 691.167,106.5 690.833,106.5 690.5,106.5C 672.032,98.7651 653.699,90.7651 635.5,82.5C 648.016,76.4095 660.683,70.5762 673.5,65C 663.506,64.5002 653.506,64.3335 643.5,64.5C 635.016,63.507 626.349,63.1736 617.5,63.5C 634.269,36.5202 658.769,22.8535 691,22.5C 723.231,22.8535 747.731,36.5202 764.5,63.5 Z"/>
<path style="opacity:1" fill="#f5fafd" d="M 736.5,56.5 C 707.662,56.8313 678.995,56.498 650.5,55.5C 654.098,53.9098 657.764,52.4098 661.5,51C 683.833,50.3333 706.167,50.3333 728.5,51C 731.407,52.6091 734.073,54.4424 736.5,56.5 Z"/>
<path style="opacity:1" fill="#eaf7fd" d="M 650.5,55.5 C 678.995,56.498 707.662,56.8313 736.5,56.5C 732.94,57.5012 728.607,57.8345 723.5,57.5C 723.5,59.5 723.5,61.5 723.5,63.5C 712.5,63.5 701.5,63.5 690.5,63.5C 675.167,63.5 659.833,63.5 644.5,63.5C 645.053,59.7208 647.053,57.0541 650.5,55.5 Z"/>
<path style="opacity:1" fill="#4b94cb" d="M 617.5,63.5 C 626.349,63.1736 635.016,63.507 643.5,64.5C 643.5,65.1667 643.167,65.5 642.5,65.5C 634.167,64.8333 625.833,64.1667 617.5,63.5 Z"/>
<path style="opacity:0.985" fill="#277abe" d="M 617.5,63.5 C 625.833,64.1667 634.167,64.8333 642.5,65.5C 640.013,70.4734 637.513,75.4734 635,80.5C 636.926,89.255 640.26,97.5884 645,105.5C 650.7,114.257 656.867,122.591 663.5,130.5C 663.569,135.854 663.902,141.187 664.5,146.5C 664.351,147.552 664.517,148.552 665,149.5C 673.707,158.704 682.207,168.037 690.5,177.5C 691.077,178.773 691.743,178.773 692.5,177.5C 699.178,166.139 705.845,154.806 712.5,143.5C 717.46,139.712 722.293,135.712 727,131.5C 727.5,122.506 727.666,113.506 727.5,104.5C 733.833,97.1667 740.167,89.8333 746.5,82.5C 746.804,81.8501 747.137,81.1835 747.5,80.5C 745.5,75.5 743.5,70.5 741.5,65.5C 749.472,64.5023 757.472,64.169 765.5,64.5C 777.293,84.3207 780.793,105.654 776,128.5C 767.539,160.962 747.372,182.462 715.5,193C 672.603,202.376 639.103,188.876 615,152.5C 602.206,127.892 600.539,102.559 610,76.5C 612.022,71.7768 614.522,67.4434 617.5,63.5 Z"/>
<path style="opacity:1" fill="#68c1ea" d="M 723.5,64.5 C 723.5,73.8333 723.5,83.1667 723.5,92.5C 713.663,97.3434 703.663,102.01 693.5,106.5C 692.833,106.5 692.5,106.833 692.5,107.5C 692.167,107.5 691.833,107.5 691.5,107.5C 691.5,107.167 691.5,106.833 691.5,106.5C 691.5,102.167 691.5,97.8333 691.5,93.5C 691.5,83.8333 691.5,74.1667 691.5,64.5C 702.167,64.5 712.833,64.5 723.5,64.5 Z"/>
<path style="opacity:1" fill="#4e8bc9" d="M 723.5,63.5 C 737.167,63.5 750.833,63.5 764.5,63.5C 765.167,63.5 765.5,63.8333 765.5,64.5C 757.472,64.169 749.472,64.5023 741.5,65.5C 743.5,70.5 745.5,75.5 747.5,80.5C 747.137,81.1835 746.804,81.8501 746.5,82.5C 738.712,85.5608 731.045,88.8941 723.5,92.5C 723.5,83.1667 723.5,73.8333 723.5,64.5C 723.5,64.1667 723.5,63.8333 723.5,63.5 Z"/>
<path style="opacity:1" fill="#a2d8f1" d="M 690.5,63.5 C 701.5,63.5 712.5,63.5 723.5,63.5C 723.5,63.8333 723.5,64.1667 723.5,64.5C 712.833,64.5 702.167,64.5 691.5,64.5C 691.5,74.1667 691.5,83.8333 691.5,93.5C 690.506,83.6806 690.173,73.6806 690.5,63.5 Z"/>
<path style="opacity:1" fill="#90d4f0" d="M 643.5,64.5 C 653.506,64.3335 663.506,64.5002 673.5,65C 660.683,70.5762 648.016,76.4095 635.5,82.5C 653.699,90.7651 672.032,98.7651 690.5,106.5C 690.141,108.371 689.141,109.871 687.5,111C 679.021,116.985 671.021,123.485 663.5,130.5C 656.867,122.591 650.7,114.257 645,105.5C 640.26,97.5884 636.926,89.255 635,80.5C 637.513,75.4734 640.013,70.4734 642.5,65.5C 643.167,65.5 643.5,65.1667 643.5,64.5 Z"/>
<path style="opacity:1" fill="#266fa8" d="M 746.5,82.5 C 740.167,89.8333 733.833,97.1667 727.5,104.5C 716.812,105.28 706.145,106.28 695.5,107.5C 694.584,107.457 693.917,107.124 693.5,106.5C 703.663,102.01 713.663,97.3434 723.5,92.5C 731.045,88.8941 738.712,85.5608 746.5,82.5 Z"/>
<path style="opacity:1" fill="#4d8ac8" d="M 727.5,104.5 C 727.666,113.506 727.5,122.506 727,131.5C 722.293,135.712 717.46,139.712 712.5,143.5C 712.167,142.833 711.833,142.167 711.5,141.5C 704.628,130.419 698.295,119.086 692.5,107.5C 692.5,106.833 692.833,106.5 693.5,106.5C 693.917,107.124 694.584,107.457 695.5,107.5C 706.145,106.28 716.812,105.28 727.5,104.5 Z"/>
<path style="opacity:1" fill="#abdcf1" d="M 690.5,106.5 C 690.833,106.5 691.167,106.5 691.5,106.5C 691.5,106.833 691.5,107.167 691.5,107.5C 691.5,115.5 691.5,123.5 691.5,131.5C 691.167,131.5 690.833,131.5 690.5,131.5C 681.502,131.119 672.502,130.786 663.5,130.5C 671.021,123.485 679.021,116.985 687.5,111C 689.141,109.871 690.141,108.371 690.5,106.5 Z"/>
<path style="opacity:1" fill="#54a5d7" d="M 691.5,107.5 C 691.833,107.5 692.167,107.5 692.5,107.5C 698.295,119.086 704.628,130.419 711.5,141.5C 704.456,138.721 697.789,135.388 691.5,131.5C 691.5,123.5 691.5,115.5 691.5,107.5 Z"/>
<path style="opacity:1" fill="#b6e0f2" d="M 663.5,130.5 C 672.502,130.786 681.502,131.119 690.5,131.5C 690.5,138.833 690.5,146.167 690.5,153.5C 683.125,152.66 675.791,151.493 668.5,150C 666.973,148.968 665.64,147.801 664.5,146.5C 663.902,141.187 663.569,135.854 663.5,130.5 Z"/>
<path style="opacity:1" fill="#58afde" d="M 690.5,131.5 C 690.833,131.5 691.167,131.5 691.5,131.5C 697.789,135.388 704.456,138.721 711.5,141.5C 711.833,142.167 712.167,142.833 712.5,143.5C 705.734,146.382 699.067,149.549 692.5,153C 691.544,156.394 691.211,159.894 691.5,163.5C 690.521,160.375 690.187,157.042 690.5,153.5C 690.5,146.167 690.5,138.833 690.5,131.5 Z"/>
<path style="opacity:1" fill="#266fa8" d="M 712.5,143.5 C 705.845,154.806 699.178,166.139 692.5,177.5C 691.743,178.773 691.077,178.773 690.5,177.5C 691.486,173.03 691.819,168.363 691.5,163.5C 691.211,159.894 691.544,156.394 692.5,153C 699.067,149.549 705.734,146.382 712.5,143.5 Z"/>
<path style="opacity:1" fill="#8fd3f0" d="M 664.5,146.5 C 665.64,147.801 666.973,148.968 668.5,150C 675.791,151.493 683.125,152.66 690.5,153.5C 690.187,157.042 690.521,160.375 691.5,163.5C 691.819,168.363 691.486,173.03 690.5,177.5C 682.207,168.037 673.707,158.704 665,149.5C 664.517,148.552 664.351,147.552 664.5,146.5 Z"/></svg><text x="471.0" y="283.0" font-size="15" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Iceberg Metadata</text><text x="471.0" y="305.0" font-size="11.5" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:Consolas,Monaco,monospace" >metadata/*.metadata.json</text><path d="M228,185 C275,185 270,121 316,121" fill="none" stroke="#5A6F77" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round"/><path d="M228,221 C275,221 270,289 316,289" fill="none" stroke="#5A6F77" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round"/><rect x="660" y="82" width="290" height="78" rx="12" fill="#FF3621" stroke="#C62828" stroke-width="2" filter="url(#uh-sh)"/><rect x="672" y="101.0" width="40" height="40" rx="9.6" fill="#fff" stroke="none" stroke-width="0" filter="url(#uh-sh)"/><svg x="678.4" y="107.4" width="27.2" height="27.2" viewBox="-19.942 -15.692 144.384 144.384" fill="none"><path d="M98.9 46.5996L52.3 72.8996L2.4 44.7996L0 46.0996V66.4996L52.3 95.8996L98.9 69.6996V80.4996L52.3 106.8L2.4 78.6996L0 79.9996V83.4996L52.3 112.9L104.5 83.4996V63.0996L102.1 61.7996L52.3 89.7996L5.6 63.5996V52.7996L52.3 78.9996L104.5 49.5996V29.4996L101.9 27.9996L52.3 55.8996L8 31.0996L52.3 6.19961L88.7 26.6996L91.9 24.8996V22.3996L52.3 0.0996094L0 29.4996V32.6996L52.3 62.0996L98.9 35.7996V46.5996Z" fill="#FF3621"/></svg><text x="831.0" y="115.0" font-size="17" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Databricks Clients</text><text x="831.0" y="137.0" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Read via Delta protocol</text><rect x="660" y="238" width="290" height="102" rx="12" fill="#455A64" stroke="#263238" stroke-width="2" filter="url(#uh-sh)"/><text x="805.0" y="272.0" font-size="17" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >External Tools</text><text x="805.0" y="294.0" font-size="13.5" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Snowflake · Trino · Athena · Spark OSS</text><text x="805.0" y="316.0" font-size="13.5" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Read via Iceberg REST Catalog</text><path d="M574,121 H656" fill="none" stroke="#1B3139" stroke-width="2.6" marker-end="url(#uh-ah-n)" stroke-linecap="round" stroke-linejoin="round"/><path d="M574,289 H656" fill="none" stroke="#1B3139" stroke-width="2.6" marker-end="url(#uh-ah-n)" stroke-linecap="round" stroke-linejoin="round"/></svg>


### Enabling Iceberg Reads 

<div style="display:flex;flex-direction:column;gap:10px;margin:16px 0;">
  <div style="display:flex;align-items:flex-start;gap:12px;background:#F5F5F5;border-radius:6px;padding:12px 14px;">
    <div style="font-size:22px;min-width:32px;">1️⃣</div>
    <div>
      <div style="font-weight:bold;font-size:13px;color:#333;">Disable Deletion Vectors</div>
      <div style="font-size:13px;color:#555;margin-top:2px;">
        <strong>Property:</strong> <code style="font-size:12px;">'delta.enableDeletionVectors' = 'false'</code><br>
        Iceberg v2 cannot represent Delta's soft-delete markers. Disabling ensures all deletes are hard deletes, making the table fully readable by Iceberg clients.
      </div>
    </div>
  </div>
  <div style="display:flex;align-items:flex-start;gap:12px;background:#F5F5F5;border-radius:6px;padding:12px 14px;">
    <div style="font-size:22px;min-width:32px;">2️⃣</div>
    <div>
      <div style="font-weight:bold;font-size:13px;color:#333;">Column Mapping Mode</div>
      <div style="font-size:13px;color:#555;margin-top:2px;">
        <strong>Property:</strong> <code style="font-size:12px;">'delta.columnMapping.mode' = 'name'</code><br>
        Ensures column identifiers are consistent between Delta and Iceberg schemas, preventing schema drift and enabling seamless cross-platform access.
      </div>
    </div>
  </div>
  <div style="display:flex;align-items:flex-start;gap:12px;background:#F5F5F5;border-radius:6px;padding:12px 14px;">
    <div style="font-size:22px;min-width:32px;">3️⃣</div>
    <div>
      <div style="font-weight:bold;font-size:13px;color:#333;">Enable IcebergCompatV2</div>
      <div style="font-size:13px;color:#555;margin-top:2px;">
        <strong>Property:</strong> <code style="font-size:12px;">'delta.enableIcebergCompatV2' = 'true'</code><br>
        Activates Delta's write protocol compatible with Iceberg v2, allowing Iceberg clients to read Delta tables without data conversion.
      </div>
    </div>
  </div>
  <div style="display:flex;align-items:flex-start;gap:12px;background:#F5F5F5;border-radius:6px;padding:12px 14px;">
    <div style="font-size:22px;min-width:32px;">4️⃣</div>
    <div>
      <div style="font-weight:bold;font-size:13px;color:#333;">Enable Universal Format</div>
      <div style="font-size:13px;color:#555;margin-top:2px;">
        <strong>Property:</strong> <code style="font-size:12px;">'delta.universalFormat.enabledFormats' = 'iceberg'</code><br>
        Triggers asynchronous Iceberg metadata generation after every Delta commit, ensuring up-to-date Iceberg views for external tools.
      </div>
    </div>
  </div>
</div>

<div style="background:#FFEBEE;border-left:4px solid #F44336;border-radius:4px;padding:14px 18px;margin:16px 0;">
  <div style="font-weight:bold;font-size:14px;color:#B71C1C;margin-bottom:6px;">⚠️ Iceberg Reads Only Work on Plain Delta Tables</div>
  <div style="font-size:14px;color:#333;">
    <strong>Note:</strong> Pipeline-managed <b>streaming tables</b> and <b>materialized views</b> cannot have Iceberg reads (UniForm with Iceberg v2) enabled. Only a plain external Delta table—such as one created via a <b>Delta Sink</b>—supports UniForm. This bridge is required for cross-platform access.
    <br><br>
    <strong>Additional Details:</strong> Setting these properties is a one-time operation, but must be done before any data is written. Once enabled, Iceberg metadata is kept in sync automatically, allowing tools like Snowflake, Trino, and Athena to query the same Delta table.
  </div>
</div>

---

<div style="background:#E3F2FD;border-left:4px solid #1976D2;padding:14px 18px;border-radius:4px;margin-top:8px;">
  <div style="font-size:16px;color:#0D47A1;line-height:1.8;">
    For more details refer to the <a href="https://docs.databricks.com/aws/en/delta/uniform" style="color:#1976D2;font-weight:bold;">Delta UniForm Documentation</a>
  </div>
</div>


#### EXPAND FOR ADDITIONAL DETAILS
<details>


<div style="font-size:16px;color:#555;line-height:2;margin-bottom:24px;">

Delta UniForm (Universal Format) solves a long-standing problem in the lakehouse ecosystem: <strong>different tools speak different table formats</strong>. Snowflake, Trino, Athena, and open-source Spark all understand Apache Iceberg — but not Delta Lake natively. UniForm bridges this gap by making your Delta tables <em>simultaneously readable as Iceberg tables</em>, without any data duplication or separate pipelines.

</div>

<div style="font-size:16px;color:#555;line-height:2;margin-bottom:24px;">

Both Delta Lake and Apache Iceberg are built on the same foundation: <strong>Parquet data files</strong> plus a <strong>metadata layer</strong>. UniForm exploits this by asynchronously generating Iceberg metadata after every Delta write — the same Parquet files now serve two formats at once. From a Delta client's perspective, nothing changes. From an Iceberg client's perspective, it looks like a native Iceberg table. You get <strong>one dataset, two logical views, zero duplication</strong>.

</div>

---

#### Benefits

<div style="font-size:16px;color:#555;line-height:2;">

- <strong>No data movement:</strong> A single copy of Parquet files is shared across Delta and Iceberg clients — no ETL, no replication, no storage overhead.
- <strong>Cross-platform interoperability:</strong> Tools like Snowflake, Trino, Apache Athena, and open-source Spark can query your Delta tables directly via the Iceberg REST Catalog.
- <strong>Negligible write overhead:</strong> Iceberg metadata generation happens asynchronously after the Delta commit, so your write pipelines are not slowed down.
- <strong>Works with Unity Catalog:</strong> Unity Catalog acts as the Iceberg REST Catalog, meaning external clients get governed, catalogued access without any extra infrastructure.

</div>

---

#### Limitations

<div style="font-size:16px;color:#555;line-height:2;">

- <strong>Read-only for Iceberg clients:</strong> External tools can only <em>read</em> via Iceberg. All writes must go through Delta — you cannot write to a UniForm table from Snowflake or Trino.
- <strong>Deletion Vectors must be disabled:</strong> Tables using deletion vectors (a Delta performance feature) cannot have UniForm enabled. You must set <code>'delta.enableDeletionVectors' = 'false'</code>. (With Iceberg v3, generally available since May 21, 2026 and requiring Databricks Runtime 18 LTS or above, this is not required.)
- <strong>Streaming Tables and Materialized Views are excluded:</strong> Pipeline-managed tables in Spark Declarative Pipelines cannot have the required <code>delta.universalFormat.enabledFormats</code> property set directly. This is where <strong>Delta Sinks</strong> come in — they write to plain external Delta tables where UniForm can be freely enabled. <em>Newer option:</em> on Databricks Runtime 17.3 and above, pipeline streaming tables and materialized views can also be exposed directly to external Iceberg v3 clients (see <a href="https://docs.databricks.com/aws/en/external-access/external-for-pipelines">External access to pipeline datasets</a>). This course uses the Delta sink pattern, which works with Iceberg v2 reads.
- <strong>Column mapping is permanent:</strong> Enabling UniForm also enables column mapping (<code>delta.columnMapping.mode = 'name'</code>), which cannot be dropped once set.
- <strong>Databricks Runtime 14.3 LTS or above required:</strong> Any Databricks client writing to a UniForm-enabled table must use DBR 14.3+.
- <strong>Table must be accessed by name:</strong> Iceberg metadata generation is only triggered when the table is accessed by name (not by path).

</div>

---

<div style="background:#E8F5E9;border-left:4px solid #2E7D32;padding:14px 20px;border-radius:4px;margin-top:8px;font-size:15px;color:#1B5E20;line-height:1.9;">
  <strong>🔗 How this connects to Delta Sinks:</strong> Since Streaming Tables and Materialized Views can't use UniForm directly, the pattern in this lecture is to use a <strong>Delta Sink</strong> — write your streaming pipeline output to a plain external Delta table, enable UniForm on that table, and Iceberg clients can immediately read it. This is the recommended production pattern for cross-platform Iceberg access from a live streaming pipeline.
</div>


## D. Connecting the Concepts

These three concepts work together to solve complex real-world data pipeline requirements.Each concept solves a distinct problem. Together they form a complete pattern for pipelines that need real-time processing and cross-platform access.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 240" width="100%" role="img" aria-label="Multiplex feeds a Delta sink, which creates the plain Delta table that enables Delta UniForm" style="font-family:'DM Sans',Arial,Helvetica,sans-serif;max-width:1000px;display:block;margin:12px auto;"><defs><filter id="ch-sh" x="-8%" y="-8%" width="116%" height="130%"><feDropShadow dx="0" dy="2" stdDeviation="3" flood-color="#0B2026" flood-opacity="0.13"/></filter><marker id="ch-ah-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#5A6F77"/></marker><marker id="ch-ah-l" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#FF3621"/></marker><marker id="ch-ah-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#2272B4"/></marker><marker id="ch-ah-n" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#1B3139"/></marker><marker id="ch-ah-k" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker></defs><rect x="10" y="30" width="230" height="190" rx="14" fill="#FF3621" stroke="#C92A18" stroke-width="2" filter="url(#ch-sh)"/><rect x="101" y="46" width="48" height="48" rx="11.52" fill="#fff" stroke="none" stroke-width="0" filter="url(#ch-sh)"/><svg x="108.68" y="55.312" width="32.64" height="29.7" viewBox="0 0 56 51" fill="#FF5F46"><rect x="14" y="6" width="34" height="3"/><rect x="14" y="6" width="3" height="33"/><rect x="45" y="6" width="3" height="19"/><rect x="14" y="17" width="34" height="3"/><rect x="24" y="20" width="3" height="17"/><rect x="14" y="37" width="17" height="3"/><path d="M32,29 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/><path d="M32,36 q2.1,-4.6 4.2,0 t4.2,0 t4.2,0 t4.2,0" fill="none" stroke="#FF5F46" stroke-width="2.6" stroke-linecap="round"/></svg><text x="125" y="124" font-size="21" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Multiplex</text><text x="125" y="158" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >One source, many event types</text><text x="125" y="182" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Ingest once — fan out by type</text><rect x="385" y="30" width="230" height="190" rx="14" fill="#2272B4" stroke="#0D47A1" stroke-width="2" filter="url(#ch-sh)"/><rect x="476" y="46" width="48" height="48" rx="11.52" fill="#fff" stroke="none" stroke-width="0" filter="url(#ch-sh)"/><svg x="483.68" y="53.68" width="32.64" height="32.64" viewBox="1.265 25.6 47.748 47.748" fill="none"><path d="M43.4295 61.3529L26.3179 31.8768C25.9325 31.2299 24.9818 31.2299 24.5964 31.8768L10.0541 56.9276C9.79712 57.3676 10.2596 57.8851 10.7221 57.6522C10.9019 57.5746 11.1332 57.4711 11.3644 57.3934C11.3644 57.3934 11.7498 57.2382 12.4435 57.0311C13.1886 56.7982 13.9851 56.5912 14.7302 56.41C16.7857 55.9442 19.4835 55.556 21.8986 55.9442C24.1339 56.3065 25.7012 56.9276 26.8317 57.5228C27.3456 57.8075 27.8338 57.1088 27.4227 56.6947C25.804 55.1161 23.2861 53.1752 19.4835 52.3729C19.3293 52.347 19.2265 52.14 19.3293 52.0106L25.0332 42.1508C25.213 41.8143 25.6755 41.8143 25.8811 42.1508L36.8264 60.9906C37.0576 61.3788 36.8521 61.8964 36.441 62.0516C35.5931 62.3363 34.5654 62.5433 33.4092 62.5433C32.6898 62.5433 31.9447 62.4657 31.1996 62.2845C31.1996 62.2845 29.0927 61.9481 26.2408 60.2401C23.4145 58.5321 17.9419 56.3583 11.8269 59.8778C9.20618 61.5599 7.45904 63.889 6.53408 65.3383C6.37992 65.5712 6.63686 65.83 6.8681 65.7006C8.97494 64.5619 14.6531 61.8964 19.7661 62.8539C22.7979 63.4232 26.2922 65.7264 29.8892 67.0721C35.8757 69.2977 41.7338 65.5712 41.7338 65.5453C42.3504 65.1312 42.7101 64.8207 42.7872 64.7689L42.8643 64.6913C43.8406 63.8632 44.0719 62.4657 43.4295 61.3529Z" fill="#00ADD4"/></svg><text x="500" y="124" font-size="21" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Delta Sink</text><text x="500" y="158" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >External Delta table</text><text x="500" y="182" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >dp.create_sink + append_flow</text><rect x="760" y="30" width="230" height="190" rx="14" fill="#2E7D32" stroke="#1B5E20" stroke-width="2" filter="url(#ch-sh)"/><rect x="851" y="46" width="48" height="48" rx="11.52" fill="#fff" stroke="none" stroke-width="0" filter="url(#ch-sh)"/><svg x="858.68" y="53.68" width="32.64" height="32.64" viewBox="579.592 -3.681 222.861 222.861" fill="none"><path style="opacity:0.973" fill="#b8e1f3" d="M 764.5,63.5 C 750.833,63.5 737.167,63.5 723.5,63.5C 723.5,61.5 723.5,59.5 723.5,57.5C 728.607,57.8345 732.94,57.5012 736.5,56.5C 734.073,54.4424 731.407,52.6091 728.5,51C 706.167,50.3333 683.833,50.3333 661.5,51C 657.764,52.4098 654.098,53.9098 650.5,55.5C 647.053,57.0541 645.053,59.7208 644.5,63.5C 659.833,63.5 675.167,63.5 690.5,63.5C 690.173,73.6806 690.506,83.6806 691.5,93.5C 691.5,97.8333 691.5,102.167 691.5,106.5C 691.167,106.5 690.833,106.5 690.5,106.5C 672.032,98.7651 653.699,90.7651 635.5,82.5C 648.016,76.4095 660.683,70.5762 673.5,65C 663.506,64.5002 653.506,64.3335 643.5,64.5C 635.016,63.507 626.349,63.1736 617.5,63.5C 634.269,36.5202 658.769,22.8535 691,22.5C 723.231,22.8535 747.731,36.5202 764.5,63.5 Z"/>
<path style="opacity:1" fill="#f5fafd" d="M 736.5,56.5 C 707.662,56.8313 678.995,56.498 650.5,55.5C 654.098,53.9098 657.764,52.4098 661.5,51C 683.833,50.3333 706.167,50.3333 728.5,51C 731.407,52.6091 734.073,54.4424 736.5,56.5 Z"/>
<path style="opacity:1" fill="#eaf7fd" d="M 650.5,55.5 C 678.995,56.498 707.662,56.8313 736.5,56.5C 732.94,57.5012 728.607,57.8345 723.5,57.5C 723.5,59.5 723.5,61.5 723.5,63.5C 712.5,63.5 701.5,63.5 690.5,63.5C 675.167,63.5 659.833,63.5 644.5,63.5C 645.053,59.7208 647.053,57.0541 650.5,55.5 Z"/>
<path style="opacity:1" fill="#4b94cb" d="M 617.5,63.5 C 626.349,63.1736 635.016,63.507 643.5,64.5C 643.5,65.1667 643.167,65.5 642.5,65.5C 634.167,64.8333 625.833,64.1667 617.5,63.5 Z"/>
<path style="opacity:0.985" fill="#277abe" d="M 617.5,63.5 C 625.833,64.1667 634.167,64.8333 642.5,65.5C 640.013,70.4734 637.513,75.4734 635,80.5C 636.926,89.255 640.26,97.5884 645,105.5C 650.7,114.257 656.867,122.591 663.5,130.5C 663.569,135.854 663.902,141.187 664.5,146.5C 664.351,147.552 664.517,148.552 665,149.5C 673.707,158.704 682.207,168.037 690.5,177.5C 691.077,178.773 691.743,178.773 692.5,177.5C 699.178,166.139 705.845,154.806 712.5,143.5C 717.46,139.712 722.293,135.712 727,131.5C 727.5,122.506 727.666,113.506 727.5,104.5C 733.833,97.1667 740.167,89.8333 746.5,82.5C 746.804,81.8501 747.137,81.1835 747.5,80.5C 745.5,75.5 743.5,70.5 741.5,65.5C 749.472,64.5023 757.472,64.169 765.5,64.5C 777.293,84.3207 780.793,105.654 776,128.5C 767.539,160.962 747.372,182.462 715.5,193C 672.603,202.376 639.103,188.876 615,152.5C 602.206,127.892 600.539,102.559 610,76.5C 612.022,71.7768 614.522,67.4434 617.5,63.5 Z"/>
<path style="opacity:1" fill="#68c1ea" d="M 723.5,64.5 C 723.5,73.8333 723.5,83.1667 723.5,92.5C 713.663,97.3434 703.663,102.01 693.5,106.5C 692.833,106.5 692.5,106.833 692.5,107.5C 692.167,107.5 691.833,107.5 691.5,107.5C 691.5,107.167 691.5,106.833 691.5,106.5C 691.5,102.167 691.5,97.8333 691.5,93.5C 691.5,83.8333 691.5,74.1667 691.5,64.5C 702.167,64.5 712.833,64.5 723.5,64.5 Z"/>
<path style="opacity:1" fill="#4e8bc9" d="M 723.5,63.5 C 737.167,63.5 750.833,63.5 764.5,63.5C 765.167,63.5 765.5,63.8333 765.5,64.5C 757.472,64.169 749.472,64.5023 741.5,65.5C 743.5,70.5 745.5,75.5 747.5,80.5C 747.137,81.1835 746.804,81.8501 746.5,82.5C 738.712,85.5608 731.045,88.8941 723.5,92.5C 723.5,83.1667 723.5,73.8333 723.5,64.5C 723.5,64.1667 723.5,63.8333 723.5,63.5 Z"/>
<path style="opacity:1" fill="#a2d8f1" d="M 690.5,63.5 C 701.5,63.5 712.5,63.5 723.5,63.5C 723.5,63.8333 723.5,64.1667 723.5,64.5C 712.833,64.5 702.167,64.5 691.5,64.5C 691.5,74.1667 691.5,83.8333 691.5,93.5C 690.506,83.6806 690.173,73.6806 690.5,63.5 Z"/>
<path style="opacity:1" fill="#90d4f0" d="M 643.5,64.5 C 653.506,64.3335 663.506,64.5002 673.5,65C 660.683,70.5762 648.016,76.4095 635.5,82.5C 653.699,90.7651 672.032,98.7651 690.5,106.5C 690.141,108.371 689.141,109.871 687.5,111C 679.021,116.985 671.021,123.485 663.5,130.5C 656.867,122.591 650.7,114.257 645,105.5C 640.26,97.5884 636.926,89.255 635,80.5C 637.513,75.4734 640.013,70.4734 642.5,65.5C 643.167,65.5 643.5,65.1667 643.5,64.5 Z"/>
<path style="opacity:1" fill="#266fa8" d="M 746.5,82.5 C 740.167,89.8333 733.833,97.1667 727.5,104.5C 716.812,105.28 706.145,106.28 695.5,107.5C 694.584,107.457 693.917,107.124 693.5,106.5C 703.663,102.01 713.663,97.3434 723.5,92.5C 731.045,88.8941 738.712,85.5608 746.5,82.5 Z"/>
<path style="opacity:1" fill="#4d8ac8" d="M 727.5,104.5 C 727.666,113.506 727.5,122.506 727,131.5C 722.293,135.712 717.46,139.712 712.5,143.5C 712.167,142.833 711.833,142.167 711.5,141.5C 704.628,130.419 698.295,119.086 692.5,107.5C 692.5,106.833 692.833,106.5 693.5,106.5C 693.917,107.124 694.584,107.457 695.5,107.5C 706.145,106.28 716.812,105.28 727.5,104.5 Z"/>
<path style="opacity:1" fill="#abdcf1" d="M 690.5,106.5 C 690.833,106.5 691.167,106.5 691.5,106.5C 691.5,106.833 691.5,107.167 691.5,107.5C 691.5,115.5 691.5,123.5 691.5,131.5C 691.167,131.5 690.833,131.5 690.5,131.5C 681.502,131.119 672.502,130.786 663.5,130.5C 671.021,123.485 679.021,116.985 687.5,111C 689.141,109.871 690.141,108.371 690.5,106.5 Z"/>
<path style="opacity:1" fill="#54a5d7" d="M 691.5,107.5 C 691.833,107.5 692.167,107.5 692.5,107.5C 698.295,119.086 704.628,130.419 711.5,141.5C 704.456,138.721 697.789,135.388 691.5,131.5C 691.5,123.5 691.5,115.5 691.5,107.5 Z"/>
<path style="opacity:1" fill="#b6e0f2" d="M 663.5,130.5 C 672.502,130.786 681.502,131.119 690.5,131.5C 690.5,138.833 690.5,146.167 690.5,153.5C 683.125,152.66 675.791,151.493 668.5,150C 666.973,148.968 665.64,147.801 664.5,146.5C 663.902,141.187 663.569,135.854 663.5,130.5 Z"/>
<path style="opacity:1" fill="#58afde" d="M 690.5,131.5 C 690.833,131.5 691.167,131.5 691.5,131.5C 697.789,135.388 704.456,138.721 711.5,141.5C 711.833,142.167 712.167,142.833 712.5,143.5C 705.734,146.382 699.067,149.549 692.5,153C 691.544,156.394 691.211,159.894 691.5,163.5C 690.521,160.375 690.187,157.042 690.5,153.5C 690.5,146.167 690.5,138.833 690.5,131.5 Z"/>
<path style="opacity:1" fill="#266fa8" d="M 712.5,143.5 C 705.845,154.806 699.178,166.139 692.5,177.5C 691.743,178.773 691.077,178.773 690.5,177.5C 691.486,173.03 691.819,168.363 691.5,163.5C 691.211,159.894 691.544,156.394 692.5,153C 699.067,149.549 705.734,146.382 712.5,143.5 Z"/>
<path style="opacity:1" fill="#8fd3f0" d="M 664.5,146.5 C 665.64,147.801 666.973,148.968 668.5,150C 675.791,151.493 683.125,152.66 690.5,153.5C 690.187,157.042 690.521,160.375 691.5,163.5C 691.819,168.363 691.486,173.03 690.5,177.5C 682.207,168.037 673.707,158.704 665,149.5C 664.517,148.552 664.351,147.552 664.5,146.5 Z"/></svg><text x="875" y="124" font-size="21" font-weight="700" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Delta UniForm</text><text x="875" y="158" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >External tools need access</text><text x="875" y="182" font-size="14" font-weight="400" fill="#fff" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Auto-generate Iceberg metadata</text><path d="M242,125 H382" fill="none" stroke="#FF7043" stroke-width="3" marker-end="url(#ch-ah-l)" stroke-linecap="round" stroke-linejoin="round"/><text x="312" y="100" font-size="13.5" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Silver table</text><text x="312" y="152" font-size="13.5" font-weight="400" fill="#5A6F77" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >feeds the sink</text><path d="M617,125 H757" fill="none" stroke="#2272B4" stroke-width="3" marker-end="url(#ch-ah-b)" stroke-linecap="round" stroke-linejoin="round"/><text x="687" y="100" font-size="13.5" font-weight="700" fill="#0B2026" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >Plain Delta table</text><text x="687" y="152" font-size="13.5" font-weight="400" fill="#5A6F77" text-anchor="middle" style="font-family:'DM Sans',Arial,Helvetica,sans-serif" >enables UniForm</text></svg>

#### EXPAND FOR ADDITIONAL DETAILS
<details>

### Combined Advanced Pipeline Pattern

#### 1. Multiplex Pattern
- **Problem**: Multiple event types in one source
- **Solution**: Single ingestion with type-based fan-out
- **Output**: Domain-specific silver tables

#### 2. Delta Sink
- **Problem**: Need external table for cross-platform access
- **Solution**: `dp.create_sink()` + `@dp.append_flow`
- **Output**: Plain Delta table outside pipeline scope

#### 3. Delta UniForm
- **Problem**: External tools need Iceberg format access  
- **Solution**: Auto-generate Iceberg metadata on plain Delta table
- **Output**: Dual-protocol table accessible by any platform
</details>



&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>